# Domino pip detector - training

Trains a small YOLO11 model that finds every pip (or pip cluster) on the table and exports it
as TFLite for the Android app. The app sums the detections, so tile colour, table colour and
touching tiles do not matter.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

Run the cells top to bottom. Cell 3 prints what the datasets contain - check it looks sane.
Cell 4 refuses to merge datasets that label different things, rather than training on
inconsistent labels. The one number that matters is printed by cell 6 (count accuracy).

## 1. Install

In [ ]:
!pip -q install ultralytics roboflow

## 2. Download the datasets
Free Roboflow account > Settings > API Keys. Paste the key below.

In [ ]:
ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'
DATASETS = [  # (workspace, project, version) - taken from the Roboflow Universe URLs
    ('elden-jahnke-snkp7', 'domino-counter', 1),
    ('pip-tracker', 'double-twelve-dominoes', 1),
]
SMOKE_TEST = False  # True = 1 epoch per model (~10 min): checks every cell works BEFORE you leave it running overnight

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
RAW = []
for ws, proj, ver in DATASETS:
    d = rf.workspace(ws).project(proj).version(ver).download('yolov8', location=f'/content/raw/{proj}')
    RAW.append(d.location)
print(RAW)

## 3. Inspect - what do the labels actually mean?

In [ ]:
import collections
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml


def split_names(root):
    return [s for s in ('train', 'valid', 'test') if os.path.isdir(f'{root}/{s}/images')]


INFO = {}
for root in RAW:
    names = yaml.safe_load(open(f'{root}/data.yaml'))['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    boxes, widths, per_image, odd_rows = collections.Counter(), [], [], 0
    for s in split_names(root):
        for lf in glob.glob(f'{root}/{s}/labels/*.txt'):
            rows = [l.split() for l in open(lf) if l.strip()]
            per_image.append(len(rows))
            for r in rows:
                if len(r) != 5:
                    odd_rows += 1
                    continue
                boxes[int(r[0])] += 1
                widths.append(float(r[3]))
    n_images = sum(len(glob.glob(f'{root}/{s}/images/*')) for s in split_names(root))
    INFO[root] = names
    print(os.path.basename(root), '| images:', n_images, '| classes:', names)
    print('   boxes per class:', {names[k]: v for k, v in sorted(boxes.items())})
    print(f'   median box width {np.median(widths):.3f} of image width | boxes per image: '
          f'median {np.median(per_image):.0f}, max {max(per_image)}')
    if odd_rows:
        print(f'   WARNING: {odd_rows} label rows are not plain boxes (polygons?) and will be ignored')

fig, axes = plt.subplots(len(RAW), 2, figsize=(14, 7 * len(RAW)), squeeze=False)
for row, root in enumerate(RAW):
    for col, p in enumerate(sorted(glob.glob(f'{root}/train/images/*'))[:2]):
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        lf = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        for l in open(lf):
            parts = l.split()
            if len(parts) != 5:
                continue
            _, x, y, bw, bh = map(float, parts)
            cv2.rectangle(im, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                          (int((x + bw / 2) * w), int((y + bh / 2) * h)), (255, 0, 0), max(2, w // 400))
        axes[row][col].imshow(im)
        axes[row][col].set_title(os.path.basename(root))
        axes[row][col].axis('off')
plt.show()

## 4. Merge into one dataset
Each class must stand for a known number of pips: `pip` is 1, and `7` or `pip-7` is a labelled
cluster of seven pips. Anything else stops here so it can be mapped by hand.

In [ ]:
import re
import shutil


def value_of(name):
    n = str(name).strip().lower()
    m = re.fullmatch(r'(?:pip[-_ ]?)?(\d+)', n)  # '7', 'pip-7', 'pip_7' -> 7
    if m:
        return int(m.group(1))
    if n == 'pip':
        return 1
    return None


kinds = {}
for root, names in INFO.items():
    vals = [value_of(n) for n in names]
    if any(v is None for v in vals):
        raise SystemExit(f'{os.path.basename(root)}: cannot tell how many pips each class stands for: '
                         f'{names}. Paste the cell-3 output back so the mapping can be set by hand.')
    kinds[root] = 'pip' if (len(names) == 1 and vals == [1]) else 'cluster'
if len(set(kinds.values())) > 1:
    raise SystemExit(f'These datasets label different things (single pips vs pip clusters): {kinds}. '
                     'Merging them would teach the model contradictory answers. Train on one kind.')

VALUES = sorted({value_of(n) for names in INFO.values() for n in names})  # pips per class id
NEW_ID = {v: i for i, v in enumerate(VALUES)}
OUT = '/content/merged'
shutil.rmtree(OUT, ignore_errors=True)
for i, (root, names) in enumerate(INFO.items()):
    for src, dst in (('train', 'train'), ('valid', 'val'), ('test', 'test')):
        if not os.path.isdir(f'{root}/{src}/images'):
            continue
        os.makedirs(f'{OUT}/{dst}/images', exist_ok=True)
        os.makedirs(f'{OUT}/{dst}/labels', exist_ok=True)
        for img in glob.glob(f'{root}/{src}/images/*'):
            stem = os.path.splitext(os.path.basename(img))[0]
            shutil.copy(img, f'{OUT}/{dst}/images/d{i}_{os.path.basename(img)}')
            lines = []
            lf = f'{root}/{src}/labels/{stem}.txt'
            if os.path.exists(lf):
                for l in open(lf):
                    p = l.split()
                    if len(p) == 5:
                        lines.append(' '.join([str(NEW_ID[value_of(names[int(p[0])])])] + p[1:]))
            open(f'{OUT}/{dst}/labels/d{i}_{stem}.txt', 'w').write('\n'.join(lines))

assert os.path.isdir(f'{OUT}/val/images'), 'no validation split found'
cfg = {'path': OUT, 'train': 'train/images', 'val': 'val/images',
       'names': ['pip'] if VALUES == [1] else [str(v) for v in VALUES]}
if os.path.isdir(f'{OUT}/test/images'):
    cfg['test'] = 'test/images'
yaml.safe_dump(cfg, open(f'{OUT}/data.yaml', 'w'))
print('pips per class id:', VALUES)
print({s: len(glob.glob(f'{OUT}/{s}/images/*')) for s in ('train', 'val', 'test')})

## 4b. Optional: add photos with NO dominoes in them
Photos of things that are not face-up dominoes (cables, boxes, empty tables...) teach the model what
to ignore. Only a slice is used for training: too many empty photos make the model timid about real
tiles. The rest are kept aside so cell 6 can measure false alarms on photos it never saw.

Run this **after cell 4** (cell 4 wipes the merged folder) and **before cell 5**.

Uploading a zip over ~200 MB from the browser is unreliable. Better: shrink the photos to ~1280 px,
or put the zip on Google Drive and set `NEG_ZIP` to its path.

In [ ]:
import random
import zipfile

MAX_TRAIN_NEGATIVES = 80  # about 10% of the labelled photos
MAX_VAL_NEGATIVES = 30
HOLDOUT = '/content/negatives_holdout'
NEG_ZIP = None  # e.g. '/content/drive/MyDrive/negatives.zip'

os.makedirs('/content/negatives', exist_ok=True)
if NEG_ZIP:
    from google.colab import drive
    drive.mount('/content/drive')
    with zipfile.ZipFile(NEG_ZIP) as z:
        z.extractall('/content/negatives')
else:
    from google.colab import files
    for name, data in files.upload().items():  # pick your negatives .zip (or the photos themselves)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('/content/negatives')
        else:
            open(f'/content/negatives/{name}', 'wb').write(data)

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp')
neg = sorted(p for p in glob.glob('/content/negatives/**/*', recursive=True) if p.lower().endswith(IMG_EXT))
random.Random(0).shuffle(neg)
n_val = min(MAX_VAL_NEGATIVES, max(1, len(neg) // 6))
n_train = min(MAX_TRAIN_NEGATIVES, len(neg) - n_val)

shutil.rmtree(HOLDOUT, ignore_errors=True)
os.makedirs(HOLDOUT)
counts = {'train': 0, 'val': 0, 'holdout': 0}
for i, p in enumerate(neg):
    split = 'val' if i < n_val else 'train' if i < n_val + n_train else 'holdout'
    im = cv2.imread(p)  # applies the photo's rotation flag, same as training does
    if im is None:
        print('skipping unreadable file:', os.path.basename(p))
        continue
    scale = 1280 / max(im.shape[:2])  # phone photos are huge; training only needs ~1280 px
    if scale < 1:
        im = cv2.resize(im, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    if split == 'holdout':
        cv2.imwrite(f'{HOLDOUT}/neg{i}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
    else:
        cv2.imwrite(f'{OUT}/{split}/images/neg{i}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{OUT}/{split}/labels/neg{i}.txt', 'w').close()  # empty label file = "nothing here"
    counts[split] += 1
print('empty photos:', counts)

## 4c. Optional but recommended overnight: keep results on Google Drive
If Colab disconnects, everything under `/content` is lost. On Drive, the checkpoints survive.
Colab asks you to approve access, so do this **before** you walk away.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/domino_runs'

## 5. Train two variants (a few hours)
Both are the same small model that runs fast on the phone. They differ only in training image size:
1280 px keeps more detail on small, far-away pips but is slower on the phone. Stronger zoom-out
augmentation (`scale=0.7`) teaches the model to read tiles that look small in the frame.
One variant failing does not stop the other. Cell 6 picks the winner.

In [ ]:
from ultralytics import YOLO

RUNS = globals().get('RUNS', '/content/runs')
EPOCHS = 1 if SMOKE_TEST else 120
VARIANTS = [('n960', 960), ('n1280', 1280)]  # (run name, training image size)

for tag, size in VARIANTS:
    try:
        YOLO('yolo11n.pt').train(
            data=f'{OUT}/data.yaml', imgsz=size, epochs=EPOCHS, patience=40, batch=-1,
            flipud=0.5, fliplr=0.5, degrees=0, scale=0.7, mosaic=1.0, close_mosaic=10,
            project=RUNS, name=tag, exist_ok=True)
    except Exception as e:
        import traceback
        traceback.print_exc()  # full error, not just its one-line message
        print(f'!! variant {tag} failed: {e!r}')

## 6. Which variant is better? Count accuracy + false alarms
mAP says how good the boxes are. What the app needs is the right *total*, so this compares predicted
against true total pips for every validation photo, at several confidence thresholds, using the same
NMS settings as the app. It also counts false alarms on the empty photos kept aside in 4b.

In [ ]:
HOLDOUT = globals().get('HOLDOUT', '/content/negatives_holdout')  # only exists if cell 4b ran
val_imgs = sorted(glob.glob(f'{OUT}/val/images/*'))
holdout_imgs = sorted(glob.glob(f'{HOLDOUT}/*.jpg')) if os.path.isdir(HOLDOUT) else []
CONFS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)


def true_total(img):
    lf = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    return sum(VALUES[int(l.split()[0])] for l in open(lf) if l.strip())


def detections(model, path, size):
    # Predict once at the lowest threshold; higher thresholds are just a filter on top, because
    # greedy NMS handles the highest scores first either way.
    r = model.predict(path, imgsz=size, conf=min(CONFS), iou=0.5, agnostic_nms=True, max_det=500, verbose=False)[0]
    return r.boxes.cls.tolist(), r.boxes.conf.tolist()


def total_at(dets, conf):
    return sum(VALUES[int(c)] for c, s in zip(*dets) if s >= conf)


truth = np.array([true_total(p) for p in val_imgs])
board = {}
for tag, size in VARIANTS:
    weights = f'{RUNS}/{tag}/weights/best.pt'
    if not os.path.exists(weights):
        print(f'{tag}: no weights (training failed?), skipping')
        continue
    model = YOLO(weights)
    val_dets = [detections(model, p, size) for p in val_imgs]
    mae = {}
    for conf in CONFS:
        err = np.array([total_at(d, conf) for d in val_dets]) - truth
        mae[conf] = float(np.mean(np.abs(err)))
        print(f'{tag} ({size}px) conf {conf}: exact {np.mean(err == 0):.0%} | within 1 pip '
              f'{np.mean(np.abs(err) <= 1):.0%} | mean abs error {mae[conf]:.2f} pips')
    best_conf = min(mae, key=mae.get)
    board[tag] = (mae[best_conf], best_conf, size)
    if holdout_imgs:
        hold = [detections(model, p, size) for p in holdout_imgs]
        alarms = sum(1 for d in hold if total_at(d, best_conf) > 0)
        print(f'{tag}: false alarms on {len(hold)} unseen empty photos at conf {best_conf}: '
              f'{alarms} ({alarms / len(hold):.0%})')

assert board, 'no variant produced weights - check the output of cell 5'
WINNER = min(board, key=lambda k: board[k][0])
BEST_ERR, BEST_CONF, BEST_SIZE = board[WINNER]
print(f'WINNER: {WINNER} at conf {BEST_CONF} (mean abs error {BEST_ERR:.2f} pips)')

## 6b. Score on YOUR real photos (the honest test)
Upload the zip of app scans, the files named like `scan_..._total14_model12.jpg`. The number after
`total` is the score you confirmed, so no boxes are needed. `model` is what the model that was in the
app at the time said, which gives the baseline to beat. These photos are never used for training.

Run this after cell 6. It replaces the winner picked in cell 6 with the one that does best on your
photos. The confidence is also chosen on them, so the number is very slightly optimistic.

In [ ]:
import re
import zipfile

REAL_ZIP = None  # e.g. '/content/drive/MyDrive/training pics.zip'
os.makedirs('/content/real', exist_ok=True)
if REAL_ZIP:
    with zipfile.ZipFile(REAL_ZIP) as z:
        z.extractall('/content/real')
else:
    from google.colab import files
    for name, data in files.upload().items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall('/content/real')
        else:
            open(f'/content/real/{name}', 'wb').write(data)

real = []
for p in sorted(glob.glob('/content/real/**/*.jpg', recursive=True)):
    m = re.search(r'total(\d+)_model(\d+)', os.path.basename(p))
    if m:
        real.append((p, int(m.group(1)), int(m.group(2))))
assert real, 'no files named like scan_..._total14_model12.jpg were found'
real_truth = np.array([tr for _, tr, _ in real])
old_err = np.array([mt - tr for _, tr, mt in real])
print(f'{len(real)} real photos with a confirmed total')
print(f'BASELINE (model that was in the app): exact {np.mean(old_err == 0):.0%} | within 1 pip '
      f'{np.mean(np.abs(old_err) <= 1):.0%} | mean abs error {np.mean(np.abs(old_err)):.2f} pips')

real_board = {}
for tag, size in VARIANTS:
    weights = f'{RUNS}/{tag}/weights/best.pt'
    if not os.path.exists(weights):
        continue
    model = YOLO(weights)
    dets = [detections(model, p, size) for p, _, _ in real]
    per_conf = {}
    for conf in CONFS:
        err = np.array([total_at(d, conf) for d in dets]) - real_truth
        per_conf[conf] = float(np.mean(np.abs(err)))
        print(f'{tag} ({size}px) conf {conf}: exact {np.mean(err == 0):.0%} | within 1 pip '
              f'{np.mean(np.abs(err) <= 1):.0%} | mean abs error {per_conf[conf]:.2f} | too high {np.mean(err > 0):.0%}')
    bc = min(per_conf, key=per_conf.get)
    real_board[tag] = (per_conf[bc], bc, size)

assert real_board, 'no trained variants found - run cell 5 first'
WINNER = min(real_board, key=lambda k: real_board[k][0])
BEST_ERR, BEST_CONF, BEST_SIZE = real_board[WINNER]
print(f'WINNER on your real photos: {WINNER} at conf {BEST_CONF} (mean abs error {BEST_ERR:.2f} pips)')

## 7. Export the winner for the app

In [ ]:
import json

best = YOLO(f'{RUNS}/{WINNER}/weights/best.pt')
best.export(format='tflite', imgsz=BEST_SIZE, half=False)
exported = glob.glob(f'{RUNS}/{WINNER}/weights/**/*.tflite', recursive=True)
print(exported)
assert exported, 'no .tflite produced - re-run this cell once (the first run installs converters)'

# Newer exporters write weights/best.tflite; older ones write best_saved_model/best_float32.tflite.
# Prefer the plain full-precision file over any float16 / int8 variant.
model_path = sorted(exported, key=lambda p: ('float16' in p or 'int8' in p, len(p)))[0]

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

interp = Interpreter(model_path=model_path)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print('MODEL :', model_path)
print('INPUT :', inp['shape'], inp['dtype'])
print('OUTPUT:', out['shape'], out['dtype'])

os.makedirs('/content/app_assets', exist_ok=True)
shutil.copy(model_path, '/content/app_assets/pip_detector.tflite')
json.dump({'classValues': VALUES, 'conf': BEST_CONF, 'iou': 0.5}, open('/content/app_assets/pip_model.json', 'w'))

from google.colab import files

for f in ('pip_detector.tflite', 'pip_model.json'):
    files.download(f'/content/app_assets/{f}')

Put both downloaded files in `app/src/main/assets/` and rebuild the app.
Paste the cell-6 lines (accuracy, WINNER, false alarms) and the MODEL / INPUT / OUTPUT lines back into the chat.